# Smoking is bad

In [ ]:
# TODO: install all the necessary packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import seaborn as sns

In [ ]:
# TODO: load the data using the pathlib library to create the path to your data
DATA_PATH = Path.cwd() / "data" / "smoking_data.csv"

df = pd.read_csv(DATA_PATH, index_col=0)
df.head()


In [ ]:
# TODO: what are the dimensions of the data frame?
df.shape

In [ ]:
# TODO: use .describe() on your data frame: what do you notice about the string columns?
df.describe()  #It has missing variables
df.describe(include="all") #It has all variables


In [ ]:
# TODO: do not forget to check the null values
df.isna().sum()

In [ ]:
# TODO: prepare your data frame so that you can work with
df["alive"] = (df["outcome"] == "Alive").astype(int)
df["smokes"] = (df["smoker"] == "Yes").astype(int)

# Find age group
df["age_group"] = pd.cut(
    df["age"],
    bins=[17, 30, 40, 50, 60, 70, 90],
    labels=["18-30", "31-40", "41-50", "51-60", "61-70", "71+"],
)

df.head()


## Smoking analysis
Let's first check what your cousin did.

In [ ]:
# TODO: Reimplement the small code snippet described in the problem set instructions.
df.groupby(["smokes"]).agg(prob=("alive",np.mean))

You should get the same results.

# Is smoking really good?
Now it is up to you to run the analysis further and check what could be underlying your cousings results.

In [ ]:
#My own analysis - Age
smoke_share = df.groupby("age_group", observed=True)["smokes"].mean()

fig, ax = plt.subplots(figsize=(8, 4))
smoke_share.plot(kind="bar", ax=ax, color="gray")
ax.set_ylabel("Share who smoke")
ax.set_xlabel("Age group")
ax.set_title("Older people are much less likely to smoke")
plt.xticks(rotation=0)
plt.show()

df.groupby("smokes")["age"].mean()

#We can see that there are less elder people smoke



In [ ]:
#age and survive time
surv_by_age = df.groupby("age_group", observed=True)["alive"].mean()

fig, ax = plt.subplots(figsize=(8, 4))
surv_by_age.plot(kind="bar", ax=ax, color="tab:blue")
ax.set_ylabel("Share alive")
ax.set_xlabel("Age group")
ax.set_title("Survival falls steeply with age")
plt.xticks(rotation=0)
plt.show()

#For elder people, their age is not that relative to smoking, as they are just getting older.

In [ ]:
#compare same age
by_group = df.pivot_table(index="age_group", columns="smokes", values="alive",
                          aggfunc="mean", observed=True)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(by_group.index, by_group[0], marker="o", label="Non-smoker")
ax.plot(by_group.index, by_group[1], marker="o", label="Smoker")
ax.set_ylabel("Share alive")
ax.set_xlabel("Age group")
ax.set_title("Survival by smoking status, within age groups")
ax.legend(loc="lower left")
plt.show()

by_group.round(3)

#In general, smoker are easier die earlier.

In [ ]:
#My own analysis - Gender
df.groupby(["gender", "smokes"])[["alive", "age"]].mean().round(3)

#visualization
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
df.groupby(["gender", "smokes"])["alive"].mean().unstack().plot(kind="bar", ax=axes[0])
axes[0].set_title("Share alive by gender and smoking")
axes[0].set_ylabel("Share alive")
df.groupby(["gender", "smokes"])["age"].mean().unstack().plot(kind="bar", ax=axes[1])
axes[1].set_title("Mean age by gender and smoking")
axes[1].set_ylabel("Mean age")
for ax in axes:
    ax.tick_params(axis="x", rotation=0)
    ax.legend(["Non-smoker", "Smoker"], loc="lower right")
plt.tight_layout()
plt.show()

# Female in general has a higher share alive. From the graph, it can easily see that smoker has a lower share alive.


In [ ]:
#My own analysis - Salary
sal = df.dropna(subset=["salary"]).copy()
sal["salary_quartile"] = pd.qcut(sal["salary"], 4, labels=["Q1", "Q2", "Q3", "Q4"])

sal.groupby("salary_quartile", observed=True)[["smokes", "alive"]].mean().round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sal.groupby("salary_quartile", observed=True)[["smokes", "alive"]].mean().plot(kind="bar", ax=ax)
ax.set_title("Smoking rate and survival by salary quartile")
ax.set_ylabel("Share")
ax.set_xlabel("Salary quartile (low to high)")
ax.legend(["Share who smoke", "Share alive"], loc="lower right")
plt.xticks(rotation=0)
plt.show()

#Salary didn't has that much relatives with share alive.